# SINCON 2026 팀 핫식스 최종 스택 재현

이 노트북은 검증 완료된 `final_package` → `final_delivery_v2`의 주후보를 입력으로 받아, 지정된 최종 후보를 단계 0~7 순서대로 재현한다. 점수 개선 실험은 하지 않는다. 기본 입력은 패키지의 보관본이다. 다른 Python 버전에서 만든 주후보도 ID·순서가 같고 기준 파일과 최대 `log1p` 차이가 `1e-9` 이하면 받는다.

보정값·창 길이·배율의 근거는 모두 **2026-06-30까지의 지하철 실측과 과거 예측 검증**이다. Public 점수는 완성된 후보 중 고르는 데만 사용했으며 이 패키지에서는 값을 다시 고르지 않는다. 행사 날짜·시각과 달력은 미래 일정도 포함한다. 보관된 시험기간 모델 예측은 실측이 아니다. 원본 모델의 외부 특징에는 시험기간 행사·날씨 정보가 들어갈 수 있으므로, 모든 정보가 6월 말 전에 공개되었다는 뜻은 아니다.

| 단계 | 무엇을 계산하며 값을 어떻게 정했나 | 기록된 과거 검증과 한계 |
|---|---|---|
| 0 주후보 입력 | `submission_night_v4_main.csv`, SHA `4c85257294be…`. 앞선 두 노트북의 출력이며 여기서는 새로 학습하지 않는다. | 앞선 패키지의 재현 검증 완료는 기존 기록이다. 이 노트북은 0번 입력의 수치 동등성을 새로 검사한다. |
| 1 K1 | 최근 84일의 과거 log 오차에서 역·날짜 평균을 빼고 역·요일·시간대별 적합. 6일 이상, `n/(n+12)` 수축, 0.25배. 제외 7개 역과 휴일·행사 보호. | 중첩 기준일 7개 중 5개 개선. Δ RMSLE는 `+0.000103, −0.000082, −0.000059, +0.000061, −0.000191, −0.000380, −0.000492`. 8월·12월 기준은 악화. |
| 2 K3 | K1에 월요일 대체공휴일 후보와 코리아컵·걷기대회 후보가 주후보와 다른 행을 값 그대로 옮긴다. 이 두 후보는 K3 정의에 명시된 보관 입력이다. | 월요일 4/4 개선, Δ `−0.000006~−0.000033`. 코리아컵 4행·걷기대회 7행은 각각 과거 사례 1건이 근거다. |
| 3 검단·PLAVE | 검단 3107의 최근 56일 일반일 오차에서 두 월 각각 12일·평균 +0.04 초과 조건. 일별 오차 중앙값을 0~0.16으로 제한해 절반 적용. 최종 약 +0.07480887, 1,872행. PLAVE는 과거 공연 시작 두 시간의 공연별 중앙 오차를 −0.30~0으로 제한해 35% 적용, 4행에 −0.105. | 검단의 날짜 비중복 3~6월 RMSLE `0.146005 → 0.145350`은 요일별 후보에 적용한 별도 평가다. 검단을 고를 때 같은 평가 일부를 참고했다. PLAVE는 과거 세 공연 12행 `0.576491 → 0.508940`이며 전체 기간 성능의 근거는 아니다. |
| 4 계절 재적합 | 기준일마다 K1을 다시 적용한 과거 예측의 잔차를 사용한다. 역·평일/주말·시간별, 연중 원형거리 92일 가중, 28 수축, 35% 적용, 프로필 ±0.12 제한. K2·검단·PLAVE 역·날짜 보호. | 계절층 자체는 8개 중 6개 개선, 2025-08-31과 12-31 악화. K3+검단+계절 조합의 날짜 비중복 A·C `0.154591 → 0.154054`, 3~6월 `0.146029 → 0.145258`. 이 전체 점수에는 시험기간 코리아컵·걷기·PLAVE 동일 행이 없다. |
| 5 L1 확장 | K1이 제외한 7개 역의 일반일에도 같은 84일·12·0.25·6일 규칙을 적용한다. 이미 K2·검단·PLAVE가 바꾼 역·날짜는 제외한다. | 중첩 기준일 8개 중 7개 개선, 평균 Δ 약 `−0.000043`, 한 기준일은 약 `+0.00003` 악화. |
| 6 curvature | 최근 70일 실측의 역·요일별 18시간 프로필. 반감기 28일, 6일 이상. DCT 주파수 6~17의 차이만 15% 적용, 최대 ±0.065, 역·날짜 log 평균 유지. | 원 기록은 기준일 9개 중 6개 개선, 평균 Δ 약 `−0.00006`. 독립 구현 기록은 같은 6개 개선·3개 악화, 평균 약 `−0.00008`. 2025-10-31, 12-31, 2026-02-28에서 악화했다. 두 날짜 비중복 계절 묶음은 두 보관 경로 모두 개선했다. |
| 7 XGBoost 차이 | 보관 `xgb_test.npy − (0.5·lgb_test.npy + 0.5·mean(base_test_s0..3.npy))`의 0.1배를 제외 7개 역의 일반일에만 log 공간에서 더한다. 157,194행의 `test.csv` 순서를 따른다. | 해당 범위 과거 9개 기준일 중 8개 개선, 평균 Δ 약 `−0.00045`. 전체 행 검증을 본 뒤 7개 역으로 좁힌 선택이며 처음부터 사전 고정된 범위는 아니다. 3107 기존 수준 보정과 신호가 일부 겹친다. 보관 npy 재학습은 기본 실행에 포함하지 않는다. |

측정 기록의 출처는 [후보 묶음 1](evidence/CANDIDATE_SET_PREREGISTRATION.md), [후보 묶음 2](evidence/CANDIDATE_SET2_PREREGISTRATION.md), [층별 점검](evidence/STACK_AUDIT_1113.md), [K 조합 검증](evidence/KSET_CODEX_COMBINATIONS.md), [검단 검증](evidence/NIGHT_V4_GEOMDAN3107.md), [PLAVE 검증](evidence/NIGHT_V4_PLAVE_IN_SHOW.md)이다. 위 점수는 보관된 과거 검증 기록이며 이 노트북이 전체 과거 평가를 다시 측정했다는 뜻은 아니다.

최종 기준 파일은 `submission_xgb7_w10_on_curvature.csv`, SHA-256 `596876180bb00eda250041cc6cf8ada4c6aa4702547f7e14a94ffe1340e3b1a6`, Public `0.144715`, 사용자 전달이다. 각 단계에서 기준 CSV와 ID·순서, 최대 log 차이 ≤1e-9, 바이트 일치를 출력한다. 바이트가 다르더라도 수치 기준을 통과하면 허용한다. 입력별 최대 날짜와 실측·일정·예측 구분도 출력한다.

CPU에서 pandas·NumPy만 사용하며 네트워크에 접속하지 않는다. 첫 코드 셀에서 읽기 전용 패키지 위치와 별도 출력 폴더를 지정할 수 있다. 실제 Jupyter 커널과 Kaggle·Colab 검증 상태는 [VERIFICATION.md](VERIFICATION.md)를 확인한다.


In [ ]:
# ===== 사용자 설정 =====
ROOT = ""             # stack_pipeline.py가 있는 패키지 폴더. 비우면 자동 검색.
BASELINE_CSV = ""     # 앞선 두 노트북에서 만든 주후보. 비우면 패키지 보관 기준 파일.
OUTPUT_DIR = ""       # 패키지 밖의 쓰기 가능한 폴더. Kaggle은 /kaggle/working 자동 선택.

In [ ]:
import glob, json, os, sys
from pathlib import Path
sys.dont_write_bytecode = True

def find_root():
    candidates = [ROOT, os.environ.get("SINCON_STACK_ROOT", ""), ".", "final_stack_delivery", "/content/final_stack_delivery"]
    candidates += sorted(glob.glob("/kaggle/input/**/stack_pipeline.py", recursive=True))
    for name in candidates:
        if not name:
            continue
        path = Path(name)
        if path.name == "stack_pipeline.py":
            path = path.parent
        if (path / "stack_pipeline.py").is_file() and (path / "SOURCE_MANIFEST.json").is_file():
            return path.resolve()
    raise FileNotFoundError("첫 코드 셀 ROOT에 패키지 폴더를 지정하세요.")

root = find_root()
requested_output = OUTPUT_DIR or os.environ.get("SINCON_STACK_OUTPUT", "")
if not requested_output:
    requested_output = "/kaggle/working/sincon_outputs" if Path("/kaggle/working").is_dir() else (
        "/content/outputs" if Path("/content").is_dir() else str(root.parent / "sincon_stack_outputs"))
out_dir = Path(requested_output).resolve()
assert out_dir != root and root not in out_dir.parents, "출력 폴더는 읽기 전용 패키지 밖에 지정하세요."
out_dir.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import hourbias_layer as hb
import stack_pipeline as stack
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| NumPy", np.__version__)
print("패키지", root, "| 출력", out_dir)
paths = [out_dir / name for name in stack.STAGES]
results, details = [], {}

## 입력 날짜 확인과 단계 0

공식 train과 두 OOF 보관본의 실측 날짜는 2026-06-30 이하여야 한다. OOF의 실측 log 값은 공식 train의 동일 키와 대조한다. 달력·행사·예측의 시험기간 날짜는 실측 날짜와 구분한다. 기준 파일을 계산값으로 덮어쓰지 않는다.

In [ ]:
ctx = stack.load_inputs(root)
test = ctx["test"]
input_audit = stack.audit_inputs(root, test)
baseline_path = Path(BASELINE_CSV or os.environ.get("SINCON_STACK_BASELINE", "") or root / "reference" / stack.STAGES[0]).resolve()
main = stack.read_prediction(baseline_path, test)
results.append(stack.verify_stage(0, baseline_path, root, test))
print("공식 학습 실측 최대 날짜", ctx["train"].date.max().date())
print("과거 예측 정답의 공식 train 일치와 기준일 < 대상 날짜 확인 완료")

## 단계 1 · K1

최근 84일 과거 오차로 일반일 시간대 모양을 적합한다. 역·날짜의 log 평균과 보호 행을 보존한다.

In [ ]:
k1, details["1_k1"] = hb.build_k1(test, main, ctx["history"], root / "cache")
hb.check_submission(k1, test)
k1.to_csv(paths[1], index=False)
results.append(stack.verify_stage(1, paths[1], root, test))
print(json.dumps(details["1_k1"], ensure_ascii=False, indent=2))

## 단계 2 · K3

원래 K3 정의대로 두 보관 후보가 주후보와 다른 행을 옮긴다. 이 노트북에서 월요일·코리아컵·걷기대회 모델을 재학습하지 않는다.

In [ ]:
monday = stack.read_prediction(root / "inputs/submission_monday_main.csv", test)
cup_walk = stack.read_prediction(root / "inputs/submission_night_v4_main_cup_walk.csv", test)
k3, details["2_k3"] = stack.build_k3(ctx, k1, monday, cup_walk)
hb.check_submission(k3, test)
k3.to_csv(paths[2], index=False)
results.append(stack.verify_stage(2, paths[2], root, test))
print(details["2_k3"])

## 단계 3 · 검단 수준과 PLAVE 공연 중 4행

검단의 과거 오차 중앙값과 PLAVE의 과거 공연별 오차를 다시 적합한다. 검단은 원래 계절·요일 모형의 제외 역이므로 그 역의 보관 원예측과 변환된 예측이 같다.

In [ ]:
details["3_geomdan_plave"] = stack.build_geomdan_plave(ctx, baseline_path, paths[2], paths[3])
results.append(stack.verify_stage(3, paths[3], root, test))
print(json.dumps(details["3_geomdan_plave"], ensure_ascii=False, indent=2))

## 단계 4 · 계절 재적합

전체 보관 OOF의 각 기준일에서 K1을 재현하고, K1 이후 남은 오차로 계절층을 적합한다. 기존 K2·검단·PLAVE 역·날짜는 보호한다.

In [ ]:
details["4_seasonal"] = stack.build_seasonal(ctx, baseline_path, paths[2], paths[3], paths[4])
results.append(stack.verify_stage(4, paths[4], root, test))
print(details["4_seasonal"])

## 단계 5 · L1 제외 7개 역 확장

K1과 같은 규칙을 제외 7개 역의 일반일에도 적용한다. 기존 휴일·행사·검단·PLAVE 변경 역·날짜는 건너뛴다.

In [ ]:
details["5_l1"] = stack.build_l1(ctx, baseline_path, paths[1], paths[3], paths[4], paths[5])
results.append(stack.verify_stage(5, paths[5], root, test))
print(details["5_l1"])

## 단계 6 · 시간대 고주파 패턴

2026-06-30까지의 공식 실측 프로필을 만들고, 고주파 성분 차이만 반영한다. 달력과 행사 보호표를 합친 원래 범위를 사용한다.

In [ ]:
details["6_curvature"] = stack.build_curvature(ctx, baseline_path, paths[5], paths[6])
results.append(stack.verify_stage(6, paths[6], root, test))
print(details["6_curvature"])

## 단계 7 · 보관 모델 예측 차이

6개 npy를 기본 입력으로 사용한다. float32 MLP 평균을 포함한 원본 NumPy 연산 순서를 유지한다. 다른 기기의 MLP 재학습은 동일 수치를 보장하지 않는다.

In [ ]:
details["7_xgb_delta"] = stack.build_xgb_delta(ctx, paths[6], paths[7])
results.append(stack.verify_stage(7, paths[7], root, test))
print(details["7_xgb_delta"])
print("최종 파일", paths[7])
print("최종 SHA-256", stack.sha256(paths[7]))
print("지정된 최종 SHA와 일치", stack.sha256(paths[7]) == stack.FINAL_SHA256)

## 실행 결과 저장

기준과의 최대 차이·ID·바이트 일치, 입력 최대 날짜, 버전을 별도 출력 폴더에 저장한다. 허용 오차를 넘는 단계에서는 즉시 중단한다.

In [ ]:
report = dict(python=sys.version.split()[0], pandas=pd.__version__, numpy=np.__version__,
              stages=results, details=details, input_audit=input_audit,
              final_sha256=stack.sha256(paths[7]), expected_final_sha256=stack.FINAL_SHA256)
(out_dir / "stack_verification.json").write_text(json.dumps(report, ensure_ascii=False, indent=2, default=int) + "\n", encoding="utf-8")
print(pd.DataFrame(results)[["stage", "max_abs_log1p_diff", "ids_and_order_match", "byte_identical"]].to_string(index=False))
print("실행 결과 저장", out_dir / "stack_verification.json")